# Uniform-sampling training-data amount sweep with retrained surrogate

This notebook is the uniformly sampled counterpart of `ml_32_corner_far_to_near.ipynb`.

Instead of holding out one geometry corner for validation/test and growing the
training set from the far corner inward, **every split is drawn uniformly at
random from across the whole parameter space**:

- The test and validation sets are a uniform random sample of the full dataset.
- The training pool is the remaining uniform random sample.
- Each training fraction (10%, 20%, ... 100%) is a nested uniform random subset
  of that training pool.

Everything else matches the corner notebook:

- The inverse-model architecture and compile hyperparameters are copied from the
  saved best combined model (`best_keras_model_surrogate_defined_loss.keras`).
- The out-of-range penalty (`qiskit_range_penalty`) is applied with the best
  optimized weight (`BEST_PENALTY_WEIGHT = 1.0`), penalizing predictions outside
  the scaled `[0, 1]` range, i.e. outside the total space spanned by the complete
  dataset.
- For each fraction, a fresh forward surrogate is retrained on the same limited
  training subset (same surrogate architecture/hyperparameters as `ml_11`) and
  then frozen while the inverse model is trained against it.

It writes:

- `data_amount_sweep_uniform_retrained_surrogate.csv`
- `data_amount_sweep_uniform_retrained_surrogate_summary.csv`
- `model/uniform_data_amount_sweep_retrained_surrogate/fraction_*_surrogate.keras`
- `model/uniform_data_amount_sweep_retrained_surrogate/fraction_*_combined.keras`
- `model/uniform_data_amount_sweep_retrained_surrogate/fraction_*_inverse.keras`

Here 100% means 100% of the (uniformly sampled) non-held-out training pool. The
uniformly sampled validation and test rows are still excluded from training.


In [1]:
from __future__ import annotations

import gc
import json
import os
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input, LeakyReLU
from tensorflow.keras.models import load_model


tf.keras.backend.set_floatx("float32")


# This notebook can be run either from the transmon experiment folder or from the
# repo root. The block below tries both so local paths do not need hard-coding.
HERE = Path.cwd()
EXPERIMENT_RELATIVE = Path("experiments/model_predict_qubit_TransmonCross_Hamiltonian_params")

if (HERE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE
elif (HERE / EXPERIMENT_RELATIVE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE / EXPERIMENT_RELATIVE
else:
    raise FileNotFoundError(
        "Could not find the transmon-cross metadata file. "
        "Run this notebook from the repo root or from the transmon experiment folder."
    )

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

from parameters_surrogate_defined_loss import (  # noqa: E402
    EPOCHS,
    MODEL_DIR as PARAM_MODEL_DIR,
    SCALERS_DIR as PARAM_SCALERS_DIR,
    TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS,
)
from parameters_surrogate import (  # noqa: E402
    EPOCHS as SURROGATE_EPOCHS,
    TRAIN_BATCH_SIZE as SURROGATE_TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE as SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS as SURROGATE_TRAIN_LOSS,
)

METADATA_DIR = EXPERIMENT_DIR / "metadata"
METADATA_PATH = METADATA_DIR / "qubit-TransmonCross-Hamiltonian_params.json"
OUT_PATH = EXPERIMENT_DIR / "data_amount_sweep_uniform_retrained_surrogate.csv"
SUMMARY_OUT_PATH = EXPERIMENT_DIR / "data_amount_sweep_uniform_retrained_surrogate_summary.csv"

MODEL_DIR = Path(PARAM_MODEL_DIR)
SCALERS_DIR = Path(PARAM_SCALERS_DIR)
SWEEP_MODEL_DIR = MODEL_DIR / "uniform_data_amount_sweep_retrained_surrogate"
SWEEP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SURROGATE_MODEL_PATH = MODEL_DIR / "best_keras_model_model2_surrogate.keras"
COMBINED_MODEL_CANDIDATES = [
    MODEL_DIR / "surrogate_loss_2in_3out_best_model.keras",
    MODEL_DIR / "best_keras_model_surrogate_defined_loss.keras",
]
COMBINED_MODEL_PATH = next((path for path in COMBINED_MODEL_CANDIDATES if path.exists()), None)
if COMBINED_MODEL_PATH is None:
    raise FileNotFoundError(f"No reference combined inverse+surrogate model found in {MODEL_DIR}")
INVERSE_MODEL_PATH = COMBINED_MODEL_PATH
ACTIVE_SURROGATE_MODEL_PATH = SURROGATE_MODEL_PATH

# Retrain a fresh surrogate for each limited training subset, then freeze it
# while training the inverse model for that same subset.
RETRAIN_SURROGATE_PER_SUBSET = True

# Ten evenly spaced training-pool fractions.
FRACTIONS = tuple(np.linspace(0.10, 1.00, 10).round(2))
SEEDS = (0,)

# Uniform random split across the whole parameter space (no held-out corner).
# Test and validation are uniform random samples of the full dataset; the rest
# is the training pool. Each fraction is a nested uniform random subset of it.
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15
SPLIT_SEED = 42

# Run the training sweep. Set this to False if the CSV already exists and you
# only want to remake the plots.
RUN_SWEEP = True

EPS = 1e-12


In [2]:
# data loading

@dataclass
class Scaler:
    min_: np.ndarray
    max_: np.ndarray

    @property
    def range_(self) -> np.ndarray:
        return np.maximum(self.max_ - self.min_, EPS)

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.min_) / self.range_

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return x * self.range_ + self.min_


def parse_um(value: object) -> float:
    text = str(value).strip()
    for suffix in ("um", "µm", "μm"):
        if text.endswith(suffix):
            return float(text[: -len(suffix)])
    return float(text)


def load_arrays() -> tuple[np.ndarray, np.ndarray]:
    """Load Hamiltonian targets and geometry values from the SQuADDS metadata."""
    data = json.loads(METADATA_PATH.read_text())
    hamiltonian = []
    geometry = []

    for row in data:
        h = row["Hamiltonian_params"]
        opts = row["design"]["design_options"]
        readout = opts["connection_pads"]["readout"]

        hamiltonian.append(
            [
                float(h["qubit_frequency_GHz"]),
                float(h["anharmonicity_MHz"]),
            ]
        )
        geometry.append(
            [
                parse_um(readout["claw_length"]),
                parse_um(readout["ground_spacing"]),
                parse_um(opts["cross_length"]),
            ]
        )

    return np.asarray(hamiltonian, dtype=np.float64), np.asarray(geometry, dtype=np.float64)


def choose_uniform_split(
    n_rows: int,
    test_fraction: float,
    val_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Draw test/validation/training-pool indices uniformly at random.

    Unlike the corner notebook, no geometry corner is held out. The test and
    validation rows are a uniform random sample from across the whole parameter
    space, and the rest becomes the training pool.
    """
    n_test = int(np.ceil(test_fraction * n_rows))
    n_val = int(np.ceil(val_fraction * n_rows))

    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_rows)

    test_idx = perm[:n_test]
    val_idx = perm[n_test:n_test + n_val]
    train_pool_idx = perm[n_test + n_val:]

    return train_pool_idx, val_idx, test_idx


def uniform_training_order(train_pool_idx: np.ndarray, seed: int) -> np.ndarray:
    """
    Return a uniform random ordering (local indices) of the training pool.

    Taking the first ``n`` of this ordering gives nested uniform random subsets
    as the training fraction grows, the analogue of the corner notebook's
    far-to-near ordering but without any spatial structure.
    """
    rng = np.random.default_rng(seed + 1000)
    return rng.permutation(len(train_pool_idx))


def scaler_from_artifacts(
    values: np.ndarray,
    columns: list[str],
    path_patterns: list[str],
    label: str,
) -> tuple[Scaler, list[str]]:
    """Load per-column MinMaxScaler ranges when present, otherwise fit on metadata."""
    mins = []
    maxs = []
    sources = []

    for i, col in enumerate(columns):
        loaded = None
        source = None
        for pattern in path_patterns:
            candidate = SCALERS_DIR / pattern.format(col=col)
            if candidate.exists():
                loaded = joblib.load(candidate)
                source = str(candidate)
                break

        if loaded is not None:
            mins.append(float(np.asarray(loaded.data_min_).reshape(-1)[0]))
            maxs.append(float(np.asarray(loaded.data_max_).reshape(-1)[0]))
            sources.append(source)
        else:
            mins.append(float(np.min(values[:, i])))
            maxs.append(float(np.max(values[:, i])))
            sources.append(f"metadata fallback: {label}.{col}")

    return Scaler(np.asarray(mins), np.asarray(maxs)), sources


In [3]:
# building the split and scalers

h_raw, geom_raw_um = load_arrays()
geom_raw_si = geom_raw_um * 1e-6

HAMILTONIAN_COLUMN_NAMES = (METADATA_DIR / "X_names").read_text().splitlines()
QISKIT_PARAM_NAMES = np.load(METADATA_DIR / "y_columns.npy", allow_pickle=True).astype(str).tolist()

train_pool_idx, val_idx, test_idx = choose_uniform_split(
    len(geom_raw_um),
    TEST_FRACTION,
    VAL_FRACTION,
    SPLIT_SEED,
)

# Make sure the held-out validation/test rows never leak into the training pool.
assert len(np.intersect1d(train_pool_idx, val_idx)) == 0
assert len(np.intersect1d(train_pool_idx, test_idx)) == 0
assert len(np.intersect1d(val_idx, test_idx)) == 0

# Uniform random ordering of the training pool; nested subsets grow with fraction.
uniform_order = uniform_training_order(train_pool_idx, SPLIT_SEED)

# For the actual model inputs/outputs, use the saved scaler artifacts from the
# existing repo workflow. These are fit on the complete dataset, so the scaled
# [0, 1] range used by the range penalty is the total space spanned by the
# complete dataset. This also keeps the surrogate in its original scaled space.
h_model_scaler, h_scaler_sources = scaler_from_artifacts(
    h_raw,
    HAMILTONIAN_COLUMN_NAMES,
    ["scaler_X_{col}.save", "scaler_X_linear_{col}.save"],
    "Hamiltonian",
)
geom_inverse_scaler, geom_inverse_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_{col}_one_hot_encoding.save"],
    "inverse_qiskit",
)
geom_surrogate_scaler, geom_surrogate_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_linear_{col}.save", "scaler_y_{col}_one_hot_encoding.save"],
    "surrogate_qiskit",
)

h_model_scaled = h_model_scaler.transform(h_raw).astype("float32")
geom_inverse_scaled = geom_inverse_scaler.transform(geom_raw_si).astype("float32")
geom_surrogate_scaled = geom_surrogate_scaler.transform(geom_raw_si).astype("float32")

# Convert inverse output scaler space to surrogate input scaler space.
# surrogate_scaled = inverse_scaled * scale_a + scale_b
scale_a = (geom_inverse_scaler.range_ / geom_surrogate_scaler.range_).astype("float32")
scale_b = ((geom_inverse_scaler.min_ - geom_surrogate_scaler.min_) / geom_surrogate_scaler.range_).astype("float32")

print(f"Loaded {len(h_raw)} total samples")
print(f"Training pool: {len(train_pool_idx)}")
print(f"Validation (uniform): {len(val_idx)}")
print(f"Test (uniform): {len(test_idx)}")
print()
print("Split check passed:")
print("  train/val overlap:", len(np.intersect1d(train_pool_idx, val_idx)))
print("  train/test overlap:", len(np.intersect1d(train_pool_idx, test_idx)))
print("  val/test overlap:", len(np.intersect1d(val_idx, test_idx)))
print("  100% means all non-held-out training-pool samples, not all samples.")
print()
print("Using saved model-space scalers from the existing repo workflow.")
for name, lo, hi, source in zip(HAMILTONIAN_COLUMN_NAMES, h_model_scaler.min_, h_model_scaler.max_, h_scaler_sources):
    print(f"  Hamiltonian {name}: {lo:.6g} to {hi:.6g}  ({source})")
for name, lo, hi, source in zip(QISKIT_PARAM_NAMES, geom_inverse_scaler.min_, geom_inverse_scaler.max_, geom_inverse_scaler_sources):
    print(f"  Inverse geometry {name}: {lo:.6g} to {hi:.6g} SI units  ({source})")
for name, lo, hi, source in zip(QISKIT_PARAM_NAMES, geom_surrogate_scaler.min_, geom_surrogate_scaler.max_, geom_surrogate_scaler_sources):
    print(f"  Surrogate geometry {name}: {lo:.6g} to {hi:.6g} SI units  ({source})")

if any(source.startswith("metadata fallback") for source in h_scaler_sources + geom_inverse_scaler_sources + geom_surrogate_scaler_sources):
    print()
    print("Note: at least one scaler artifact was not found, so metadata-derived min/max ranges were used for that column.")


Loaded 1934 total samples
Training pool: 1352
Validation (uniform): 291
Test (uniform): 291

Split check passed:
  train/val overlap: 0
  train/test overlap: 0
  val/test overlap: 0
  100% means all non-held-out training-pool samples, not all samples.

Using saved model-space scalers from the existing repo workflow.
  Hamiltonian qubit_frequency_GHz: 3.21853 to 7.12659  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_qubit_frequency_GHz.save)
  Hamiltonian anharmonicity_MHz: -525.818 to -88.9577  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_anharmonicity_MHz.save)
  Inverse geometry design_options.connection_pads.readout.claw_length: 7e-05 to 0.0004 SI units  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_y_design_options.connection_pads.readout.claw_length_one_hot_encoding.save)
  Inverse ge

In [4]:

def read_keras_config(path: Path) -> dict:
    with zipfile.ZipFile(path) as zf:
        return json.loads(zf.read("config.json"))


def extract_reference_specs(combined_path: Path) -> tuple[dict, dict]:
    cfg = read_keras_config(combined_path)
    layers = cfg["config"]["layers"]
    inverse_cfg = next(
        layer for layer in layers
        if layer.get("class_name") == "Sequential" and layer.get("config", {}).get("name") == "inverse_model"
    )
    compile_cfg = cfg.get("compile_config") or {}
    return inverse_cfg["config"], compile_cfg


def extract_surrogate_specs(surrogate_path: Path) -> tuple[dict, dict]:
    cfg = read_keras_config(surrogate_path)
    if cfg.get("class_name") != "Sequential":
        raise ValueError(f"Expected a Sequential surrogate model, got {cfg.get('class_name')} from {surrogate_path}")
    return cfg["config"], cfg.get("compile_config") or {}


def _initializer_from_config(config: dict | None, seed: int | None):
    if not config:
        return None
    cfg = json.loads(json.dumps(config))
    if seed is not None and isinstance(cfg.get("config"), dict) and "seed" in cfg["config"]:
        cfg["config"]["seed"] = seed
    try:
        return tf.keras.initializers.deserialize(cfg)
    except Exception:
        class_name = cfg.get("class_name")
        if class_name == "HeNormal":
            return tf.keras.initializers.HeNormal(seed=cfg.get("config", {}).get("seed"))
        if class_name == "LecunUniform":
            return tf.keras.initializers.LecunUniform(seed=cfg.get("config", {}).get("seed"))
        if class_name == "Zeros":
            return tf.keras.initializers.Zeros()
        return tf.keras.initializers.get(class_name)


def _regularizer_from_config(config: dict | None):
    if not config:
        return None
    try:
        return tf.keras.regularizers.deserialize(config)
    except Exception:
        if config.get("class_name") == "L2":
            return tf.keras.regularizers.l2(config.get("config", {}).get("l2", 0.01))
        raise


def build_sequential_from_config(config: dict, input_dim: int, seed: int, default_name: str) -> Sequential:
    tf.keras.utils.set_random_seed(seed)
    model = Sequential(name=config.get("name", default_name))
    for layer_idx, layer_cfg in enumerate(config["layers"]):
        class_name = layer_cfg["class_name"]
        cfg = layer_cfg["config"]
        if class_name == "InputLayer":
            model.add(Input(shape=(input_dim,), name=cfg.get("name", "input")))
        elif class_name == "Dense":
            model.add(
                Dense(
                    cfg["units"],
                    activation=cfg.get("activation", "linear"),
                    name=cfg.get("name"),
                    kernel_initializer=_initializer_from_config(cfg.get("kernel_initializer"), seed + layer_idx),
                    bias_initializer=_initializer_from_config(cfg.get("bias_initializer"), None),
                    kernel_regularizer=_regularizer_from_config(cfg.get("kernel_regularizer")),
                    bias_regularizer=_regularizer_from_config(cfg.get("bias_regularizer")),
                )
            )
        elif class_name == "LeakyReLU":
            model.add(LeakyReLU(negative_slope=cfg.get("negative_slope", 0.01), name=cfg.get("name")))
        elif class_name == "Dropout":
            model.add(Dropout(rate=cfg.get("rate", 0.0), name=cfg.get("name")))
        else:
            raise ValueError(f"Unsupported reference layer type: {class_name}")
    return model


def build_inverse_from_reference(input_dim: int, seed: int) -> Sequential:
    return build_sequential_from_config(REFERENCE_INVERSE_CONFIG, input_dim, seed, "inverse_model")


def build_surrogate_from_reference(input_dim: int, seed: int) -> Sequential:
    model = build_sequential_from_config(SURROGATE_REFERENCE_CONFIG, input_dim, seed, "retrained_surrogate")
    model.compile(
        optimizer=build_surrogate_optimizer(),
        loss=SURROGATE_RECONSTRUCTION_LOSS,
        metrics=[SURROGATE_TRAIN_LOSS],
        jit_compile=SURROGATE_JIT_COMPILE,
    )
    return model


class ScalerConversionLayer(tf.keras.layers.Layer):
    def __init__(self, scale_a, scale_b, **kwargs):
        kwargs.setdefault("trainable", False)
        super().__init__(**kwargs)
        self._scale_a = tf.constant(scale_a, dtype=tf.float32)
        self._scale_b = tf.constant(scale_b, dtype=tf.float32)
        self._cfg = {"scale_a": list(np.asarray(scale_a, dtype=float)), "scale_b": list(np.asarray(scale_b, dtype=float))}

    def call(self, inputs):
        a = tf.cast(self._scale_a, inputs.dtype)
        b = tf.cast(self._scale_b, inputs.dtype)
        return inputs * a + b

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config


def qiskit_range_penalty(y_true_dummy, y_pred):
    """Penalize inverse predictions outside the scaled [0, 1] training range."""
    below = tf.nn.relu(-y_pred)
    above = tf.nn.relu(y_pred - 1.0)
    return tf.reduce_mean(below ** 2 + above ** 2)


def load_frozen_surrogate(surrogate_model_path: Path):
    surrogate_model = load_model(surrogate_model_path, compile=False)
    surrogate_model.trainable = False
    for layer in surrogate_model.layers:
        layer.trainable = False
    return surrogate_model


def deserialize_reference_learning_rate(lr_config):
    if isinstance(lr_config, (int, float, np.integer, np.floating)):
        return float(lr_config)
    if isinstance(lr_config, str):
        try:
            return float(lr_config)
        except ValueError:
            return lr_config
    if isinstance(lr_config, dict):
        cfg = lr_config.get("config", {})
        for key in ("value", "initial_value"):
            if key in cfg and isinstance(cfg[key], (int, float, np.integer, np.floating)):
                return float(cfg[key])
        try:
            return tf.keras.optimizers.schedules.deserialize(lr_config)
        except Exception:
            if "initial_learning_rate" in cfg:
                return tf.keras.optimizers.schedules.ExponentialDecay(
                    initial_learning_rate=float(cfg["initial_learning_rate"]),
                    decay_steps=cfg.get("decay_steps", 1),
                    decay_rate=cfg.get("decay_rate", 1.0),
                    staircase=cfg.get("staircase", False),
                )
            raise
    return lr_config


def describe_learning_rate(lr) -> str:
    if isinstance(lr, (int, float, np.integer, np.floating)):
        return str(float(lr))
    if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
        return f"{lr.__class__.__name__}({lr.get_config()})"
    return str(lr)


def optimizer_lr_label(compile_config: dict) -> str:
    try:
        lr_cfg = compile_config["optimizer"]["config"]["learning_rate"]
        return describe_learning_rate(deserialize_reference_learning_rate(lr_cfg))
    except Exception:
        return "unknown"


def build_reference_optimizer():
    try:
        return tf.keras.optimizers.deserialize(REFERENCE_COMPILE_CONFIG["optimizer"])
    except Exception:
        return tf.keras.optimizers.Adam(learning_rate=REFERENCE_LEARNING_RATE)


def build_surrogate_optimizer():
    try:
        return tf.keras.optimizers.deserialize(SURROGATE_COMPILE_CONFIG["optimizer"])
    except Exception:
        return tf.keras.optimizers.Adam()


def build_combined_model(seed: int, surrogate_model_path: Path) -> tuple[Sequential, Model]:
    tf.keras.backend.clear_session()
    surrogate_model = load_frozen_surrogate(surrogate_model_path)
    inverse_model = build_inverse_from_reference(h_model_scaled.shape[1], seed=seed)

    combined_input = Input(shape=(h_model_scaled.shape[1],), name="combined_input")
    predicted_qiskit = inverse_model(combined_input)
    predicted_qiskit_converted = ScalerConversionLayer(scale_a, scale_b, name="scaler_conversion")(predicted_qiskit)
    reconstructed_hamiltonian = surrogate_model(predicted_qiskit_converted)

    combined_model = Model(
        inputs=combined_input,
        outputs=[reconstructed_hamiltonian, predicted_qiskit],
        name="combined_model",
    )

    combined_model.compile(
        optimizer=build_reference_optimizer(),
        loss=[REFERENCE_RECONSTRUCTION_LOSS, qiskit_range_penalty],
        loss_weights=REFERENCE_LOSS_WEIGHTS,
        jit_compile=REFERENCE_JIT_COMPILE,
    )

    return inverse_model, combined_model


REFERENCE_INVERSE_CONFIG, REFERENCE_COMPILE_CONFIG = extract_reference_specs(COMBINED_MODEL_PATH)
SURROGATE_REFERENCE_CONFIG, SURROGATE_COMPILE_CONFIG = extract_surrogate_specs(SURROGATE_MODEL_PATH)

REFERENCE_OPTIMIZER_CONFIG = REFERENCE_COMPILE_CONFIG["optimizer"]["config"]
REFERENCE_LEARNING_RATE = deserialize_reference_learning_rate(REFERENCE_OPTIMIZER_CONFIG["learning_rate"])
REFERENCE_LEARNING_RATE_LABEL = describe_learning_rate(REFERENCE_LEARNING_RATE)
REFERENCE_RECONSTRUCTION_LOSS = REFERENCE_COMPILE_CONFIG["loss"][0]
REFERENCE_LOSS_WEIGHTS = [float(v) for v in REFERENCE_COMPILE_CONFIG.get("loss_weights", [1.0, 1.0])]
# Pin the out-of-range (qiskit_range_penalty) weight to the best optimized value
# from the ml_21 Keras-tuner search. The best trial (val loss 0.001002) used
# penalty_weight = 1.0, matching the saved best combined model loss_weights.
# Setting it explicitly keeps this sweep identical to the best model regardless
# of which reference model file is loaded.
BEST_PENALTY_WEIGHT = 1.0
REFERENCE_LOSS_WEIGHTS = [REFERENCE_LOSS_WEIGHTS[0], BEST_PENALTY_WEIGHT]
REFERENCE_JIT_COMPILE = bool(REFERENCE_COMPILE_CONFIG.get("jit_compile", False))
REFERENCE_BATCH_SIZE = TRAIN_BATCH_SIZE
REFERENCE_EPOCHS = EPOCHS
REFERENCE_EARLY_STOPPING_PATIENCE = TRAIN_EARLY_STOPPING_PATIENCE

SURROGATE_RECONSTRUCTION_LOSS = SURROGATE_COMPILE_CONFIG.get("loss", SURROGATE_TRAIN_LOSS)
SURROGATE_LEARNING_RATE_LABEL = optimizer_lr_label(SURROGATE_COMPILE_CONFIG)
SURROGATE_JIT_COMPILE = bool(SURROGATE_COMPILE_CONFIG.get("jit_compile", False))

reference_dense_layers = [
    layer["config"]["units"]
    for layer in REFERENCE_INVERSE_CONFIG["layers"]
    if layer["class_name"] == "Dense"
]
surrogate_dense_layers = [
    layer["config"]["units"]
    for layer in SURROGATE_REFERENCE_CONFIG["layers"]
    if layer["class_name"] == "Dense"
]
print("Reference inverse dense units:", reference_dense_layers)
print("Reference inverse learning rate:", REFERENCE_LEARNING_RATE_LABEL)
print("Reference inverse reconstruction loss:", REFERENCE_RECONSTRUCTION_LOSS)
print("Reference inverse loss weights:", REFERENCE_LOSS_WEIGHTS)
print("Reference inverse jit_compile:", REFERENCE_JIT_COMPILE)
print("Inverse epochs/batch/patience:", REFERENCE_EPOCHS, REFERENCE_BATCH_SIZE, REFERENCE_EARLY_STOPPING_PATIENCE)
print()
print("Reference surrogate dense units:", surrogate_dense_layers)
print("Reference surrogate learning rate:", SURROGATE_LEARNING_RATE_LABEL)
print("Reference surrogate loss:", SURROGATE_RECONSTRUCTION_LOSS)
print("Reference surrogate jit_compile:", SURROGATE_JIT_COMPILE)
print("Surrogate epochs/batch/patience:", SURROGATE_EPOCHS, SURROGATE_TRAIN_BATCH_SIZE, SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE)


Reference inverse dense units: [64, 3]
Reference inverse learning rate: ExponentialDecay({'initial_learning_rate': 0.001, 'decay_steps': 220, 'decay_rate': 0.99, 'staircase': False, 'name': 'ExponentialDecay'})
Reference inverse reconstruction loss: mae
Reference inverse loss weights: [1.0, 1.0]
Reference inverse jit_compile: True
Inverse epochs/batch/patience: 400 128 60

Reference surrogate dense units: [736, 2]
Reference surrogate learning rate: 0.05013880506157875
Reference surrogate loss: mae
Reference surrogate jit_compile: True
Surrogate epochs/batch/patience: 400 128 60


In [5]:
def fit_surrogate_for_subset(
    geom_subset_scaled: np.ndarray,
    h_subset_scaled: np.ndarray,
    geom_val_scaled: np.ndarray,
    h_val_scaled: np.ndarray,
    seed: int,
) -> tuple[Sequential, tf.keras.callbacks.History]:
    surrogate_model = build_surrogate_from_reference(geom_subset_scaled.shape[1], seed=seed)
    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0,
    )
    history = surrogate_model.fit(
        np.asarray(geom_subset_scaled, dtype="float32"),
        np.asarray(h_subset_scaled, dtype="float32"),
        epochs=SURROGATE_EPOCHS,
        batch_size=SURROGATE_TRAIN_BATCH_SIZE,
        validation_data=(
            np.asarray(geom_val_scaled, dtype="float32"),
            np.asarray(h_val_scaled, dtype="float32"),
        ),
        callbacks=[early_stopping],
        verbose=0,
    )
    return surrogate_model, history


def fit_inverse_for_subset(
    h_subset_scaled: np.ndarray,
    h_val_scaled: np.ndarray,
    seed: int,
    surrogate_model_path: Path,
) -> tuple[Sequential, Model, tf.keras.callbacks.History]:
    inverse_model, combined_model = build_combined_model(seed=seed, surrogate_model_path=surrogate_model_path)
    dummy_train = np.zeros((len(h_subset_scaled), len(QISKIT_PARAM_NAMES)), dtype="float32")
    dummy_val = np.zeros((len(h_val_scaled), len(QISKIT_PARAM_NAMES)), dtype="float32")

    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=REFERENCE_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0,
    )

    history = combined_model.fit(
        np.asarray(h_subset_scaled, dtype="float32"),
        [np.asarray(h_subset_scaled, dtype="float32"), dummy_train],
        epochs=REFERENCE_EPOCHS,
        batch_size=REFERENCE_BATCH_SIZE,
        validation_data=(
            np.asarray(h_val_scaled, dtype="float32"),
            [np.asarray(h_val_scaled, dtype="float32"), dummy_val],
        ),
        callbacks=[early_stopping],
        verbose=0,
    )

    return inverse_model, combined_model, history


def evaluate_percent_error(
    combined_model: Model,
    h_scaled_in: np.ndarray,
    h_unscaled: np.ndarray,
) -> dict[str, float]:
    h_pred_scaled, _ = combined_model.predict(np.asarray(h_scaled_in, dtype="float32"), verbose=0)
    h_pred = h_model_scaler.inverse_transform(h_pred_scaled)
    pct = 100.0 * np.abs(h_pred - h_unscaled) / np.maximum(np.abs(h_unscaled), EPS)

    return {
        "fq_mean_pct": float(np.mean(pct[:, 0])),
        "alpha_mean_pct": float(np.mean(pct[:, 1])),
        "mean_hamiltonian_pct": float(np.mean(pct)),
    }


def evaluate_surrogate_model(
    surrogate_model: Model,
    geom_scaled_in: np.ndarray,
    h_unscaled: np.ndarray,
) -> dict[str, float]:
    h_pred_scaled = surrogate_model.predict(np.asarray(geom_scaled_in, dtype="float32"), verbose=0)
    h_pred = h_model_scaler.inverse_transform(h_pred_scaled)
    pct = 100.0 * np.abs(h_pred - h_unscaled) / np.maximum(np.abs(h_unscaled), EPS)
    return {
        "fq_mean_pct": float(np.mean(pct[:, 0])),
        "alpha_mean_pct": float(np.mean(pct[:, 1])),
        "mean_hamiltonian_pct": float(np.mean(pct)),
    }


def inverse_range_stats(inverse_model: Sequential, h_scaled_in: np.ndarray) -> dict[str, float]:
    qiskit_scaled = inverse_model.predict(np.asarray(h_scaled_in, dtype="float32"), verbose=0)
    below = np.maximum(-qiskit_scaled, 0.0)
    above = np.maximum(qiskit_scaled - 1.0, 0.0)
    violation = below + above
    return {
        "qiskit_scaled_min": float(np.min(qiskit_scaled)),
        "qiskit_scaled_max": float(np.max(qiskit_scaled)),
        "qiskit_range_violation_mean": float(np.mean(violation)),
        "qiskit_range_violation_max": float(np.max(violation)),
    }


In [6]:
# surrogate training mode

print("Retraining a fresh surrogate for each fraction/seed using the same subset as the inverse model.")
print("Surrogate reference model:", SURROGATE_MODEL_PATH)
print("Range-penalty weight (best optimized):", REFERENCE_LOSS_WEIGHTS[1])
print("Sweep model output dir:", SWEEP_MODEL_DIR)


Retraining a fresh surrogate for each fraction/seed using the same subset as the inverse model.
Surrogate reference model: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/best_keras_model_model2_surrogate.keras
Range-penalty weight (best optimized): 1.0
Sweep model output dir: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/uniform_data_amount_sweep_retrained_surrogate


In [7]:
# run sweep
def model_paths_for_fraction_seed(fraction: float, seed: int) -> tuple[Path, Path, Path]:
    pct = int(round(fraction * 100))
    stem = f"fraction_{pct:03d}pct_seed{seed}"
    return (
        SWEEP_MODEL_DIR / f"{stem}_surrogate.keras",
        SWEEP_MODEL_DIR / f"{stem}_combined.keras",
        SWEEP_MODEL_DIR / f"{stem}_inverse.keras",
    )


subset_indices_by_fraction = {}

if RUN_SWEEP:
    print(f"Retraining surrogate for each subset from reference: {SURROGATE_MODEL_PATH}")
    print(f"Copying inverse architecture/compile hyperparameters from: {COMBINED_MODEL_PATH}")

    rows = []
    n_train_pool = len(train_pool_idx)
    hidden_units = [
        layer["config"]["units"]
        for layer in REFERENCE_INVERSE_CONFIG["layers"]
        if layer["class_name"] == "Dense"
    ]
    surrogate_hidden_units = [
        layer["config"]["units"]
        for layer in SURROGATE_REFERENCE_CONFIG["layers"]
        if layer["class_name"] == "Dense"
    ]

    for fraction in FRACTIONS:
        n_subset = max(1, int(round(fraction * n_train_pool)))
        subset_local = uniform_order[:n_subset]
        subset_idx = train_pool_idx[subset_local]
        subset_indices_by_fraction[fraction] = subset_idx

        print(f"Training surrogate+inverse using {fraction:.0%} of training pool ({n_subset} samples)...")

        for seed in SEEDS:
            surrogate_model_path, combined_model_path, inverse_model_path = model_paths_for_fraction_seed(fraction, seed)

            surrogate_model, surrogate_history = fit_surrogate_for_subset(
                geom_surrogate_scaled[subset_idx],
                h_model_scaled[subset_idx],
                geom_surrogate_scaled[val_idx],
                h_model_scaled[val_idx],
                seed=seed,
            )
            surrogate_model.save(surrogate_model_path)

            surrogate_train_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[subset_idx],
                h_raw[subset_idx],
            )
            surrogate_val_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[val_idx],
                h_raw[val_idx],
            )
            surrogate_test_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[test_idx],
                h_raw[test_idx],
            )

            del surrogate_model
            tf.keras.backend.clear_session()
            gc.collect()

            inverse_model, combined_model, history = fit_inverse_for_subset(
                h_model_scaled[subset_idx],
                h_model_scaled[val_idx],
                seed=seed,
                surrogate_model_path=surrogate_model_path,
            )
            combined_model.save(combined_model_path)
            inverse_model.save(inverse_model_path)

            train_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[subset_idx],
                h_raw[subset_idx],
            )
            val_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[val_idx],
                h_raw[val_idx],
            )
            test_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[test_idx],
                h_raw[test_idx],
            )
            test_range_stats = inverse_range_stats(inverse_model, h_model_scaled[test_idx])

            rows.append(
                {
                    "selection_method": "uniform_random_across_parameter_space_retrained_surrogate",
                    "split_seed": SPLIT_SEED,
                    "fraction": fraction,
                    "training_percent": fraction * 100.0,
                    "n_samples": n_subset,
                    "seed": seed,
                    "surrogate_reference_model_path": str(SURROGATE_MODEL_PATH),
                    "surrogate_training_mode": "retrained_on_same_subset_as_inverse",
                    "combined_reference_model_path": str(COMBINED_MODEL_PATH),
                    "inverse_reference_model_path": str(INVERSE_MODEL_PATH),
                    "saved_surrogate_model_path": str(surrogate_model_path),
                    "saved_combined_model_path": str(combined_model_path),
                    "saved_inverse_model_path": str(inverse_model_path),
                    "surrogate_dense_units": json.dumps(surrogate_hidden_units),
                    "inverse_dense_units": json.dumps(hidden_units),
                    "surrogate_optimizer": "Adam",
                    "surrogate_learning_rate": SURROGATE_LEARNING_RATE_LABEL,
                    "surrogate_reconstruction_loss": SURROGATE_RECONSTRUCTION_LOSS,
                    "surrogate_jit_compile": SURROGATE_JIT_COMPILE,
                    "surrogate_epochs_run": len(surrogate_history.history.get("loss", [])),
                    "surrogate_batch_size": SURROGATE_TRAIN_BATCH_SIZE,
                    "surrogate_early_stopping_patience": SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
                    "inverse_optimizer": "Adam",
                    "inverse_learning_rate": REFERENCE_LEARNING_RATE_LABEL,
                    "inverse_reconstruction_loss": REFERENCE_RECONSTRUCTION_LOSS,
                    "range_penalty_weight": REFERENCE_LOSS_WEIGHTS[1],
                    "inverse_jit_compile": REFERENCE_JIT_COMPILE,
                    "inverse_epochs_run": len(history.history.get("loss", [])),
                    "inverse_batch_size": REFERENCE_BATCH_SIZE,
                    "inverse_early_stopping_patience": REFERENCE_EARLY_STOPPING_PATIENCE,
                    **{f"surrogate_train_{key}": value for key, value in surrogate_train_metrics.items()},
                    **{f"surrogate_val_{key}": value for key, value in surrogate_val_metrics.items()},
                    **{f"surrogate_test_{key}": value for key, value in surrogate_test_metrics.items()},
                    **{f"train_{key}": value for key, value in train_metrics.items()},
                    **{f"val_{key}": value for key, value in val_metrics.items()},
                    **{f"test_{key}": value for key, value in test_metrics.items()},
                    **{f"test_{key}": value for key, value in test_range_stats.items()},
                }
            )

            del inverse_model, combined_model, history, surrogate_history
            tf.keras.backend.clear_session()
            gc.collect()

    out = pd.DataFrame(rows)
    out.to_csv(OUT_PATH, index=False)
    print()
    print(f"wrote {OUT_PATH}")
    print(f"saved sweep models to {SWEEP_MODEL_DIR}")
else:
    print(f"Skipping training. Reading existing results from {OUT_PATH}")
    out = pd.read_csv(OUT_PATH)

out.head()


Retraining surrogate for each subset from reference: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/best_keras_model_model2_surrogate.keras
Copying inverse architecture/compile hyperparameters from: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/surrogate_loss_2in_3out_best_model.keras
Training surrogate+inverse using 10% of training pool (135 samples)...


I0000 00:00:1780106795.867396 1149998 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1157 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe MIG 2g.20gb, pci bus id: 0000:00:10.0, compute capability: 8.0
I0000 00:00:1780106797.941967 1150208 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Training surrogate+inverse using 20% of training pool (270 samples)...
Training surrogate+inverse using 30% of training pool (406 samples)...
Training surrogate+inverse using 40% of training pool (541 samples)...
Training surrogate+inverse using 50% of training pool (676 samples)...
Training surrogate+inverse using 60% of training pool (811 samples)...
Training surrogate+inverse using 70% of training pool (946 samples)...
Training surrogate+inverse using 80% of training pool (1082 samples)...
Training surrogate+inverse using 90% of training pool (1217 samples)...
Training surrogate+inverse using 100% of training pool (1352 samples)...

wrote /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/data_amount_sweep_uniform_retrained_surrogate.csv
saved sweep models to /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/uniform_data_amount_sweep_retrained_surrogate


,selection_method,split_seed,fraction,training_percent,n_samples,seed,surrogate_reference_model_path,surrogate_training_mode,combined_reference_model_path,inverse_reference_model_path,...,val_fq_mean_pct,val_alpha_mean_pct,val_mean_hamiltonian_pct,test_fq_mean_pct,test_alpha_mean_pct,test_mean_hamiltonian_pct,test_qiskit_scaled_min,test_qiskit_scaled_max,test_qiskit_range_violation_mean,test_qiskit_range_violation_max
0,uniform_random_across_parameter_space_retraine...,42,0.1,10.0,135,0,/home/olivias/ML_qubit_design/experiments/mode...,retrained_on_same_subset_as_inverse,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.456373,1.114171,0.785272,0.418119,1.036220,0.727169,-0.013354,0.958386,0.000426,0.013354
1,uniform_random_across_parameter_space_retraine...,42,0.2,20.0,270,0,/home/olivias/ML_qubit_design/experiments/mode...,retrained_on_same_subset_as_inverse,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.235801,0.758640,0.497221,0.237721,0.743722,0.490722,0.031220,0.952834,0.000000,0.000000
2,uniform_random_across_parameter_space_retraine...,42,0.3,30.0,406,0,/home/olivias/ML_qubit_design/experiments/mode...,retrained_on_same_subset_as_inverse,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.120004,0.447043,0.283523,0.090352,0.510115,0.300234,-0.452768,1.390880,0.002142,0.452768
3,uniform_random_across_parameter_space_retraine...,42,0.4,40.0,541,0,/home/olivias/ML_qubit_design/experiments/mode...,retrained_on_same_subset_as_inverse,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.143291,0.347962,0.245627,0.132795,0.345689,0.239242,-0.028477,0.991555,0.002406,0.028477
4,uniform_random_across_parameter_space_retraine...,42,0.5,50.0,676,0,/home/olivias/ML_qubit_design/experiments/mode...,retrained_on_same_subset_as_inverse,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.096944,0.521051,0.308997,0.104416,0.515673,0.310044,-0.032497,0.982258,0.000155,0.032497


In [8]:
summary = (
    out.groupby(["training_percent", "n_samples"], as_index=False)
    .agg(
        train_mean=("train_mean_hamiltonian_pct", "mean"),
        train_std=("train_mean_hamiltonian_pct", "std"),
        val_mean=("val_mean_hamiltonian_pct", "mean"),
        val_std=("val_mean_hamiltonian_pct", "std"),
        test_mean=("test_mean_hamiltonian_pct", "mean"),
        test_std=("test_mean_hamiltonian_pct", "std"),
    )
    .sort_values("training_percent")
)
summary.to_csv(SUMMARY_OUT_PATH, index=False)
print(f"wrote {SUMMARY_OUT_PATH}")
summary


wrote /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/data_amount_sweep_uniform_retrained_surrogate_summary.csv


,training_percent,n_samples,train_mean,train_std,val_mean,val_std,test_mean,test_std
0,10.0,135,0.723835,NaN,0.785272,NaN,0.727169,NaN
1,20.0,270,0.471623,NaN,0.497221,NaN,0.490722,NaN
2,30.0,406,0.261670,NaN,0.283523,NaN,0.300234,NaN
3,40.0,541,0.244637,NaN,0.245627,NaN,0.239242,NaN
4,50.0,676,0.306801,NaN,0.308997,NaN,0.310044,NaN
5,60.0,811,0.403698,NaN,0.437424,NaN,0.395513,NaN
6,70.0,946,0.380822,NaN,0.398393,NaN,0.375516,NaN
7,80.0,1082,0.461299,NaN,0.482756,NaN,0.473501,NaN
8,90.0,1217,0.197307,NaN,0.204115,NaN,0.190352,NaN
9,100.0,1352,0.492604,NaN,0.515396,NaN,0.479151,NaN
